In [ ]:
!pip install -q bitsandbytes peft transformers accelerate

In [1]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Configurazione Base (Zephyr 4-bit per risparmiare RAM)
MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Caricamento Zephyr Base...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Zephyr Base caricato.")

# 2. Cerca e Carica l'Adattatore PPO
adapter_path = None
# Cerca la cartella che contiene 'adapter_config.json' nei dataset di input
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files:
        adapter_path = root
        print(f"Trovati pesi PPO in: {adapter_path}")
        break

if adapter_path:
    model = PeftModel.from_pretrained(base_model, adapter_path)
    print("MODELLO PPO V2 CARICATO CON SUCCESSO!")
else:
    print("ERRORE: Non trovo i pesi.")

2026-02-14 10:22:10.446734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771064530.469018     118 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771064530.475776     118 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771064530.493294     118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771064530.493322     118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771064530.493325     118 computation_placer.cc:177] computation placer alr

Caricamento Zephyr Base...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Zephyr Base caricato.
Trovati pesi PPO in: /kaggle/input/model-ppo-v2/zephyr-7b-ppo-borda-v2
MODELLO PPO V2 CARICATO CON SUCCESSO!


In [2]:
import random
def generate_response(user_prompt, temperature=0.8):
    
    style_dad_jokes = (
        "You are a professional comedy writer specializing in one-liners. "
        "Your goal is to make the user laugh with a SINGLE, sharp punchline. "
        "Do not ramble. Do not offer multiple options. Just one joke."
    )
    
    style_comedian = (
        "You are a witty stand-up comedian with a cynical style. "
        "Make ONE sarcastic observation or tell ONE mini-story about the topic. "
        "Do not use the Q&A format. Be direct and stop after the punchline."
    )
    
    current_system_prompt = random.choice([style_dad_jokes, style_comedian])
    # Formattazione Zephyr
    messages = [
        {"role": "system", "content": current_system_prompt},
        {"role": "user", "content":  user_prompt}
    ]
    
    # Applica il template corretto
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    # Generazione
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100, 
            do_sample=True,
            temperature=temperature,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decodifica e pulizia
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Estraiamo solo la parte dell'assistente (la battuta)
    # Zephyr di solito mette la risposta dopo l'ultimo header
    if "assistant" in generated_text:
        joke = generated_text.split("assistant")[-1].strip()
    else:
        # Fallback se il template si rompe
        joke = generated_text.replace(input_text, "").strip()
        
    return joke

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- INTERFACCIA CHATBOT DINAMICO ---

# 1. Creazione Widget
text_input = widgets.Text(
    placeholder='Scrivi un prompt (es: Tell me a joke about pizza)...',
    layout=widgets.Layout(width='70%')
)

submit_button = widgets.Button(
    description="Invia", 
    button_style="primary", # Colore blu
    icon='paper-plane'
)

output_area = widgets.Output(layout={
    'border': '1px solid #444', 
    'padding': '15px', 
    'margin': '10px 0'
})

# 2. Logica del Click
def on_submit(_):
    prompt = text_input.value
    if not prompt: return
    
    # Pulisce la casella di testo per il prossimo input
    text_input.value = "" 
    
    with output_area:
        clear_output() # Cancella la risposta precedente
        
        # Stampa il tuo messaggio
        print(f" TU: {prompt}")
        print(" Zephyr sta pensando...")
        
        # --- QUI AVVIENE LA MAGIA ---
        # Chiama la funzione generate_response che abbiamo appena modificato.
        # Sarà lei a decidere se usare lo stile "Secco" o "Storyteller".
        try:
            response = generate_response(prompt, temperature=0.8)
            
            # Pulisce "sta pensando" e mostra il risultato
            clear_output()
            print(f" TU: {prompt}")
            print(f" ZEPHYR: {response}")
            
        except Exception as e:
            print(f" Errore: {e}")

# 3. Collega gli eventi
submit_button.on_click(on_submit)
text_input.on_submit(on_submit)

# 4. Mostra tutto
print(" CHATBOT PRONTO! (Prompt Dinamico Attivo: Q&A o Story)")
display(widgets.HBox([text_input, submit_button]), output_area)

 CHATBOT PRONTO! (Prompt Dinamico Attivo: Q&A o Story)


/tmp/ipykernel_118/2062197775.py:55: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_submit)


Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…